# 2. Fusion-only GNN Ablation — Colab Full Dataset

This notebook trains the **fusion_only** ablation on the full WebNLG dataset using the saved BART baseline.

## Table of Contents
1. Setup and Google Drive
2. Write shared implementation module
3. Configuration
4. Copy/load baseline artifacts and rebuild graphs if needed
5. Build model
5.5 Previous output dashboard
6. Train on full dataset
7. Generation evaluation
8. Results and output locations

**Default mode:** full dataset. To debug quickly, set `DEBUG_MODE=True` in Section 3.
This EASY_SHARED version loads `fixed_ablation_common.py` directly from the shared Google Drive folder.


## 1. Setup and Google Drive

In [ ]:
# Install general dependencies.
!pip -q install transformers datasets nltk rouge-score accelerate sentencepiece sacremoses

import os, sys, subprocess, json, pickle, random, shutil, time
import numpy as np
import torch
from pathlib import Path

from google.colab import drive
drive.mount('/content/drive')

print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('CUDA:', torch.version.cuda)
    print('GPU:', torch.cuda.get_device_name(0))
    print('Capability:', torch.cuda.get_device_capability(0))
else:
    print('WARNING: No GPU found. Runtime > Change runtime type > GPU is strongly recommended.')

### 1.1 Install PyTorch Geometric

In [ ]:
# Install PyTorch Geometric matching the current Torch/CUDA build.
try:
    import torch_geometric
    print('torch_geometric already installed:', torch_geometric.__version__)
except Exception:
    torch_version = torch.__version__.split('+')[0]
    cuda_version = torch.version.cuda
    if cuda_version is None:
        pyg_url = f'https://data.pyg.org/whl/torch-{torch_version}+cpu.html'
    else:
        cuda_tag = 'cu' + cuda_version.replace('.', '')
        pyg_url = f'https://data.pyg.org/whl/torch-{torch_version}+{cuda_tag}.html'
    print('Installing PyG from:', pyg_url)
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install', '-q',
        'torch-geometric', 'torch-scatter', 'torch-sparse', '-f', pyg_url
    ])
    import torch_geometric
    print('Installed torch_geometric:', torch_geometric.__version__)

## 2. Load shared implementation module from Drive

This notebook expects `fixed_ablation_common.py` to be in the same shared project folder as the notebooks.

For collaborators: open the shared Drive folder or add it as a shortcut to `MyDrive`, then only update `PROJECT_DIR` if your path is different.



In [ ]:
# Simple shared-module loader.
# This replaces the old huge "write fixed_ablation_common.py" cell and the separate true-resume patch cell.

from google.colab import drive
drive.mount('/content/drive')

import os, sys, shutil, importlib

# Change this only if your shared Drive folder is mounted somewhere else.
PROJECT_DIR = "/content/drive/MyDrive/kg_llm_project"
COMMON_SRC = os.path.join(PROJECT_DIR, "fixed_ablation_common.py")
COMMON_DST = "/content/fixed_ablation_common.py"

print("PROJECT_DIR:", PROJECT_DIR)
print("Common module source:", COMMON_SRC)
print("Exists:", os.path.exists(COMMON_SRC))

if not os.path.exists(COMMON_SRC):
    raise FileNotFoundError(
        "Cannot find fixed_ablation_common.py.\n"
        "Put fixed_ablation_common.py in the shared kg_llm_project folder, "
        "or edit PROJECT_DIR in this cell."
    )

# Copy from Drive into the runtime so imports are stable and fast.
shutil.copy(COMMON_SRC, COMMON_DST)

if "/content" not in sys.path:
    sys.path.insert(0, "/content")

# Clear old cached modules in case Colab previously imported an older version.
for mod in ["fixed_ablation_common", "true_resume_patch"]:
    if mod in sys.modules:
        del sys.modules[mod]

import fixed_ablation_common as fac

print("Imported module from:", fac.__file__)
print("Has run_train_resume:", hasattr(fac, "run_train_resume"))
print("Has load_variant_checkpoint_resume:", hasattr(fac, "load_variant_checkpoint_resume"))
print("Has build_trainable_optimizer:", hasattr(fac, "build_trainable_optimizer"))

from fixed_ablation_common import *
print("Shared module loaded successfully.")



## 3. Configuration

In [ ]:
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

PROJECT_DIR = '/content/drive/MyDrive/kg_llm_project'
BASE_DATASET = f'{PROJECT_DIR}/baseline-bart-webnlg'
OUTPUT_DIR = f'{PROJECT_DIR}/fusion_only_outputs'
PROCESSED_DIR = f'{BASE_DATASET}/processed'
CHECKPOINT_DIR = f'{OUTPUT_DIR}/checkpoints'

# This is the only BART source used in the new direction.
# We do not load the old fine-tuned BART baseline checkpoint.
PRETRAINED_MODEL_NAME = 'facebook/bart-base'

for d in [PROJECT_DIR, BASE_DATASET, OUTPUT_DIR, PROCESSED_DIR, CHECKPOINT_DIR]:
    os.makedirs(d, exist_ok=True)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# Full dataset by default. Change DEBUG_MODE=True for quick testing.
DEBUG_MODE = False
MAX_TRAIN_EXAMPLES = 1000 if DEBUG_MODE else None
MAX_DEV_EXAMPLES = 200 if DEBUG_MODE else None
MAX_TEST_EXAMPLES = 200 if DEBUG_MODE else None

# Training settings. On T4, use BATCH_SIZE=16 or BATCH_SIZE=8 + GRAD_ACCUM=2.
# On A100, BATCH_SIZE=32 or 64 may fit.
EPOCHS = 3
BATCH_SIZE = 32
GRAD_ACCUM = 1

# Separate learning rates because BART is pretrained while KG modules are randomly initialized.
BART_LR = 3e-5
KG_LR = 1e-4

# Loss / decoding hyperparameters.
BETA = 0.0
GAMMA = 0.3
ALPHA = 0.0

EVAL_SPLIT = 'dev'  # use 'test' only for final reporting
MAX_GEN_LEN = 128

# Resume controls.
# First run: RESUME_TRAINING=False, EPOCHS=3.
# Continue to epoch 5 later: RESUME_TRAINING=True, EPOCHS=5.
RESUME_TRAINING = False
RESUME_CHECKPOINT = f'{CHECKPOINT_DIR}/fusion_only/model_final.pt'

# Evaluation controls.
# Set RUN_EVALUATION=True after training if you want the notebook to evaluate automatically.
RUN_EVALUATION = True
EVALUATE_CHECKPOINTS = ['best', 'final']  # can also use only ['best']

print('BASE_DATASET:', BASE_DATASET)
print('PROCESSED_DIR:', PROCESSED_DIR)
print('OUTPUT_DIR:', OUTPUT_DIR)
print('CHECKPOINT_DIR:', CHECKPOINT_DIR)
print('PRETRAINED_MODEL_NAME:', PRETRAINED_MODEL_NAME)
print('DEVICE:', DEVICE)
print('DEBUG_MODE:', DEBUG_MODE)
print('EPOCHS:', EPOCHS)
print('BATCH_SIZE:', BATCH_SIZE, 'GRAD_ACCUM:', GRAD_ACCUM, 'effective:', BATCH_SIZE * GRAD_ACCUM)
print('BART_LR:', BART_LR, 'KG_LR:', KG_LR)
print('BETA:', BETA, 'GAMMA:', GAMMA, 'ALPHA:', ALPHA)


## 4. Copy/load baseline artifacts and rebuild graphs if needed

In [ ]:
# Load static processed dataset artifacts. We do NOT load a fine-tuned BART checkpoint here.
# The model starts from pretrained facebook/bart-base and BART remains trainable.
required_files = [
    f'{PROCESSED_DIR}/webnlg_processed.pkl',
    f'{PROCESSED_DIR}/vocabularies.pkl',
]
missing = [p for p in required_files if not os.path.exists(p)]
if missing:
    raise FileNotFoundError(
        'Missing processed dataset artifacts. Expected files under PROCESSED_DIR. Missing: ' + str(missing)
    )

from transformers import BartTokenizer, BartForConditionalGeneration
from torch_geometric.data import Data
from fixed_ablation_common import load_pretrained_bart

# Trainable pretrained BART. No fine-tuned baseline checkpoint is required.
tokenizer, bart_model = load_pretrained_bart(
    PRETRAINED_MODEL_NAME,
    device=DEVICE,
    train_bart=True,
)

with open(f'{PROCESSED_DIR}/webnlg_processed.pkl', 'rb') as f:
    raw_data_for_graphs = pickle.load(f)
with open(f'{PROCESSED_DIR}/vocabularies.pkl', 'rb') as f:
    vocab_for_graphs = pickle.load(f)

relation_vocab = vocab_for_graphs['relation_vocab']

def _get_triples(ex):
    return ex['triples']

def _normalize_triple(t):
    if isinstance(t, dict):
        return str(t['subject']), str(t['predicate']), str(t['object'])
    return str(t[0]), str(t[1]), str(t[2])

@torch.no_grad()
def _initialize_node_features(entity_names, tokenizer, bart_model):
    emb = bart_model.model.shared
    device = emb.weight.device
    features = []
    for name in entity_names:
        clean = str(name).replace('_', ' ')
        toks = tokenizer(clean, return_tensors='pt', add_special_tokens=False, truncation=True, max_length=32)
        input_ids = toks['input_ids'].to(device)
        if input_ids.numel() == 0:
            pooled = torch.zeros(emb.embedding_dim, device=device)
        else:
            pooled = emb(input_ids).squeeze(0).mean(dim=0)
        features.append(pooled.detach().cpu())
    if not features:
        return torch.zeros((0, emb.embedding_dim), dtype=torch.float)
    return torch.stack(features, dim=0)

def _triples_to_graph(ex, relation_vocab, tokenizer, bart_model):
    triples = [_normalize_triple(t) for t in _get_triples(ex)]
    entities = []
    for s, p, o in triples:
        if s not in entities: entities.append(s)
        if o not in entities: entities.append(o)
    local_idx = {e:i for i,e in enumerate(entities)}
    src, dst, etypes = [], [], []
    num_rel = len(relation_vocab)
    for s,p,o in triples:
        if p not in relation_vocab: continue
        si, oi = local_idx[s], local_idx[o]
        ri = relation_vocab[p]
        src.append(si); dst.append(oi); etypes.append(ri)
        src.append(oi); dst.append(si); etypes.append(ri + num_rel)
    edge_index = torch.tensor([src, dst], dtype=torch.long) if src else torch.empty((2,0), dtype=torch.long)
    edge_type = torch.tensor(etypes, dtype=torch.long) if etypes else torch.empty((0,), dtype=torch.long)
    x = _initialize_node_features(entities, tokenizer, bart_model)
    g = Data(x=x, edge_index=edge_index, edge_type=edge_type, num_nodes=len(entities))
    g.entity_names = entities
    return g

def rebuild_graph_files_if_missing(processed_dir, tokenizer, bart_model):
    needed = [f'{processed_dir}/graphs_train.pkl', f'{processed_dir}/graphs_dev.pkl', f'{processed_dir}/graphs_test.pkl']
    if all(os.path.exists(p) for p in needed):
        print('Graph files already exist. Reusing cached graph files.')
        return
    print('Graph files missing. Rebuilding graph files from processed data.')
    for split in ['train', 'dev', 'test']:
        graphs = []
        split_data = raw_data_for_graphs[split]
        for i, ex in enumerate(split_data):
            graphs.append(_triples_to_graph(ex, relation_vocab, tokenizer, bart_model))
            if (i + 1) % 1000 == 0:
                print(f'  {split}: built {i+1}/{len(split_data)}')
        out_path = f'{processed_dir}/graphs_{split}.pkl'
        with open(out_path, 'wb') as f:
            pickle.dump(graphs, f)
        print(f'Saved {split} graphs -> {out_path} ({len(graphs)} graphs)')

rebuild_graph_files_if_missing(PROCESSED_DIR, tokenizer, bart_model)
print('Processed dir contains:', os.listdir(PROCESSED_DIR))
print('Loaded trainable pretrained BART:', PRETRAINED_MODEL_NAME)


## 5. Build model and select full dataset

In [ ]:
from fixed_ablation_common import (
    load_artifacts, attach_entity_labels, FusionOnlyGNNModel, run_generation_eval
)

data, graphs, vocab = load_artifacts(PROCESSED_DIR)
print('Loaded splits:', {k: len(v) for k, v in data.items()})
print('Graph splits:', {k: len(v) for k, v in graphs.items()})
print('Relations:', len(vocab['relation_vocab']))

def maybe_slice(xs, n):
    return xs if n is None else xs[:n]

data_run = {
    'train': maybe_slice(data['train'], MAX_TRAIN_EXAMPLES),
    'dev': maybe_slice(data['dev'], MAX_DEV_EXAMPLES),
    'test': maybe_slice(data['test'], MAX_TEST_EXAMPLES),
}
graphs_run = {
    'train': maybe_slice(graphs['train'], MAX_TRAIN_EXAMPLES),
    'dev': maybe_slice(graphs['dev'], MAX_DEV_EXAMPLES),
    'test': maybe_slice(graphs['test'], MAX_TEST_EXAMPLES),
}

attach_entity_labels(data_run, graphs_run)

num_relations = len(vocab['relation_vocab']) * 2
model = FusionOnlyGNNModel(bart_model, num_relations=num_relations).to(DEVICE)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print('Running data sizes:', {k: len(v) for k, v in data_run.items()})
print('Running graph sizes:', {k: len(v) for k, v in graphs_run.items()})
print('Trainable parameters:', f'{trainable:,} / total {total:,}')
print(model.__class__.__name__)


## 6. Train on full dataset

In [ ]:
import os
import json
import glob
import torch
import pandas as pd

def show_previous_outputs(output_dir, checkpoint_dir, variant, split="dev"):
    variant_ckpt_dir = os.path.join(checkpoint_dir, variant)
    eval_dir = os.path.join(output_dir, "evaluations")

    print("========== Previous Output Dashboard ==========")
    print("OUTPUT_DIR:", output_dir)
    print("CHECKPOINT_DIR:", checkpoint_dir)
    print("VARIANT:", variant)
    print("EVAL_SPLIT:", split)

    print("\n--- Checkpoints ---")
    best_path = os.path.join(variant_ckpt_dir, "model_best.pt")
    final_path = os.path.join(variant_ckpt_dir, "model_final.pt")

    for name, path in [("best", best_path), ("final", final_path)]:
        print(f"{name} checkpoint:", path)
        print("exists:", os.path.exists(path))

        if os.path.exists(path):
            try:
                ckpt = torch.load(path, map_location="cpu")
                if isinstance(ckpt, dict):
                    print("epoch:", ckpt.get("epoch", "unknown"))
                    print("best_val_total:", ckpt.get("best_val_total", "unknown"))
                    print("keys:", list(ckpt.keys()))
            except Exception as e:
                print("Could not inspect checkpoint:", e)

        print()

    print("\n--- Training history ---")
    history_csv = os.path.join(variant_ckpt_dir, "history.csv")
    history_json = os.path.join(variant_ckpt_dir, "history.json")

    print("history.csv exists:", os.path.exists(history_csv))
    print("history.json exists:", os.path.exists(history_json))

    if os.path.exists(history_csv):
        try:
            hist_df = pd.read_csv(history_csv)
            display(hist_df)
        except Exception as e:
            print("Could not display history.csv:", e)

    elif os.path.exists(history_json):
        try:
            with open(history_json, "r") as f:
                history = json.load(f)
            hist_df = pd.DataFrame(history)
            display(hist_df)
        except Exception as e:
            print("Could not display history.json:", e)

    print("\n--- Evaluation files ---")
    if os.path.exists(eval_dir):
        eval_files = sorted(glob.glob(os.path.join(eval_dir, "*")))
        if eval_files:
            for f in eval_files:
                print(os.path.basename(f))
        else:
            print("No evaluation files found yet.")
    else:
        print("Evaluation directory does not exist yet:", eval_dir)

    print("==============================================")

In [ ]:
VARIANT = 'fusion_only'
resume_path = RESUME_CHECKPOINT if RESUME_TRAINING else None

model, history, checkpoint_paths = run_train_resume(
    model=model,
    data=data_run,
    graphs=graphs_run,
    tokenizer=tokenizer,
    bart_model=bart_model,
    checkpoint_dir=CHECKPOINT_DIR,
    variant=VARIANT,
    device=DEVICE,
    num_epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    bart_lr=BART_LR,
    kg_lr=KG_LR,
    grad_accum=GRAD_ACCUM,
    beta=BETA,
    gamma=GAMMA,
    resume_from_checkpoint=resume_path,
    load_best_at_end=False,
)

# Save a simple copy of history under OUTPUT_DIR.
with open(f'{OUTPUT_DIR}/history_{VARIANT}.json', 'w') as f:
    json.dump(history, f, indent=2)
print('Saved history:', f'{OUTPUT_DIR}/history_{VARIANT}.json')
print('Checkpoint paths:', checkpoint_paths)


# Show updated training/checkpoint history after this training cell finishes.
show_previous_outputs(OUTPUT_DIR, CHECKPOINT_DIR, VARIANT, split=EVAL_SPLIT)


## 7. Generation evaluation

In [ ]:

VARIANT = 'fusion_only'
global_entity_set = vocab.get('global_entity_set') or list(vocab.get('entity_vocab', {}).keys())

from datetime import datetime
import os, json, torch

EVAL_OUTPUT_DIR = os.path.join(OUTPUT_DIR, 'evaluations')
os.makedirs(EVAL_OUTPUT_DIR, exist_ok=True)

if not RUN_EVALUATION:
    print('RUN_EVALUATION=False, so evaluation is skipped.')
    print('To evaluate after training, set RUN_EVALUATION=True and rerun this cell.')
    show_previous_outputs(OUTPUT_DIR, CHECKPOINT_DIR, VARIANT, split=EVAL_SPLIT)
else:
    all_eval_results = {}
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')

    for ckpt_tag in EVALUATE_CHECKPOINTS:
        if ckpt_tag == 'best':
            ckpt_path = f'{CHECKPOINT_DIR}/{VARIANT}/model_best.pt'
        elif ckpt_tag == 'final':
            ckpt_path = f'{CHECKPOINT_DIR}/{VARIANT}/model_final.pt'
        else:
            raise ValueError(f'Unknown checkpoint tag: {ckpt_tag}')

        if not os.path.exists(ckpt_path):
            print(f'Skipping {ckpt_tag} because checkpoint does not exist: {ckpt_path}')
            continue

        ckpt = torch.load(ckpt_path, map_location='cpu')
        ckpt_epoch = ckpt.get('epoch', 'unknown') if isinstance(ckpt, dict) else 'unknown'
        eval_tag = f'{VARIANT}_{ckpt_tag}_ep{ckpt_epoch}_{timestamp}'
        print('\nEvaluating:', eval_tag)

        # Build a fresh model for each checkpoint so best/final evaluations are independent.
        eval_model = FusionOnlyGNNModel(bart_model, num_relations=num_relations).to(DEVICE)
        eval_model = load_variant_checkpoint_resume(eval_model, ckpt_path, DEVICE, strict=False)

        metrics, predictions, per_sample_rows = run_generation_eval(
            model=eval_model,
            data=data_run,
            graphs=graphs_run,
            tokenizer=tokenizer,
            processed_dir=EVAL_OUTPUT_DIR,
            device=DEVICE,
            variant=VARIANT,          # keep the real variant name for generation behavior
            global_entity_set=global_entity_set,
            split=EVAL_SPLIT,
            max_gen_len=MAX_GEN_LEN,
            alpha=ALPHA,
            output_tag=eval_tag,      # versioned filenames, so old results are preserved
        )
        all_eval_results[eval_tag] = metrics

    summary_path = os.path.join(EVAL_OUTPUT_DIR, f'metrics_summary_{VARIANT}_{timestamp}_{EVAL_SPLIT}.json')
    with open(summary_path, 'w') as f:
        json.dump(all_eval_results, f, indent=2)

    print('All evaluation results from this run:')
    print(json.dumps(all_eval_results, indent=2))
    print('Saved summary:', summary_path)

    # Show all old + new results together.
    show_previous_outputs(OUTPUT_DIR, CHECKPOINT_DIR, VARIANT, split=EVAL_SPLIT)


## 8. Results and output locations

In [ ]:
print('Variant:', 'fusion_only')
print('OUTPUT_DIR:', OUTPUT_DIR)
print('CHECKPOINT_DIR:', CHECKPOINT_DIR)
print('Files in OUTPUT_DIR:')
print(os.listdir(OUTPUT_DIR))
print('Checkpoint folder:')
print(os.listdir(CHECKPOINT_DIR))

# Compact dashboard of all currently saved training/evaluation outputs.
show_previous_outputs(OUTPUT_DIR, CHECKPOINT_DIR, VARIANT, split=EVAL_SPLIT)
